<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Cross Validation</b>
</h1>
<div style="font-family:'Times New Roman';">
Every time i did a train test split i picked some random_state and trusted the one number that came out. But that number depends on which rows happened to land in the test set, so its kind of a gamble. Cross validation fixes this by testing on every part of the data and averaging, which gives a way more trustworthy score.
</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target
print(X.shape)

(569, 30)


## one split is a gamble

Let me show the problem first. I'll do the same train test split but with a bunch of diffrent random seeds, and just watch the test accuracy bounce around.

In [3]:
accs = []
for seed in range(15):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xtr, ytr)
    accs.append((m.predict(Xte) == yte).mean())

print('lowest :', round(min(accs), 3))
print('highest:', round(max(accs), 3))
print('spread :', round(max(accs) - min(accs), 3))

lowest : 0.918
highest: 0.982
spread : 0.064


So just by changing the split, the accuracy swings by a few percent. Thats too much wobble to trust a single number.

## k-fold cross validation

The idea is simple. Split the data into k equal folds (say 5). Then train on 4 folds and test on the 1 left out, and rotate so every fold gets a turn as the test set. At the end you average the 5 scores. Every point gets used for testing exactly once, so nothing is wasted and the score is much steadier.

let me build it from scratch first.

In [4]:
def k_fold_scores(model, X, y, k=5, seed=0):
    idx = np.arange(len(y))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)

    scores = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        model.fit(X[train_idx], y[train_idx])
        scores.append((model.predict(X[test_idx]) == y[test_idx]).mean())
    return np.array(scores)

In [5]:
scores = k_fold_scores(RandomForestClassifier(n_estimators=100, random_state=0), X, y, k=5)
print('fold scores:', scores.round(3))
print('mean:', scores.mean().round(3), ' std:', scores.std().round(3))

fold scores: [0.947 0.947 0.982 0.982 0.938]
mean: 0.96  std: 0.019


<h2 style="color:#1f77b4; font-family:'Times New Roman';">
<b>The sklearn way</b>
</h2>
<div style="font-family:'Times New Roman';">
Ofcourse sklearn has this built in with cross_val_score, one line and done. It should land in the same place as our from scratch version.
</div>

In [7]:
sk_scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0), X, y, cv=5)
print('sklearn folds:', sk_scores.round(3))
print('mean:', sk_scores.mean().round(3))

sklearn folds: [0.93  0.947 0.991 0.974 0.973]
mean: 0.963


## stratified k-fold

One catch with plain k-fold, if the classes are imbalanced a random fold might end up with barely any of the rare class. **Stratified** k-fold makes sure each fold keeps roughly the same class balance as the full data, which is why its the default for classification in sklearn. For most classification work you just want stratified.

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
strat_scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0), X, y, cv=skf)
print('stratified mean:', strat_scores.mean().round(3))

stratified mean: 0.965


### takeaways

- a single train test split gives a noisy score that depends on the random seed
- k-fold trains and tests on every part of the data and averages, so its steadier
- you also get a std across folds, which tells you how stable the model is
- use stratified k-fold for classification so the class balance stays the same in each fold

next i'll use cross validation inside hyperparameter tuning, which is what it was really built for.